Importation des bibliothèques

In [560]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import torch.optim as optim
from torch.utils.data import random_split
from tqdm import tqdm # pour la progression 
import time #pour le temps de calcul
import copy #pour copier base_model

In [561]:
print(os.getcwd())  # donne le répertoire courant

d:\INRIA\MCDropout


Configuration de base

Importation du modèle déjà entraîné par l'utilisateur

In [562]:
batch_size = 128 #on peut changer la taille mais 128 est bien pour éviter underfitting ou overfitting

transform = transforms.Compose([
    transforms.ToTensor(),                   #Convertir une image en tenseur, met les valeurs des pixels entre 0 et 1
    transforms.Normalize((0.5, 0.5, 0.5),    # Moyenne RGB
                         (0.5, 0.5, 0.5))])  # Écart-type RGB; pixels deviennent centrés autour de 0

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                         download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                        download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                         shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')


Définition du CNN de base (3 couches)

In [563]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)  # entrée 3 channels (RGB)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        
        self.pool = nn.MaxPool2d(2, 2)  # réduit dimension par 2 à chaque fois
        
        # Calculer la taille en sortie avant les couches fully connected
        # CIFAR10 32x32 → après 3 poolings (x2) → 4x4
        self.fc1 = nn.Linear(64 * 4 * 4, 128) # après les convolutions on a un tenseur de taille 64*4*4, fc1 les réduit en 128 neurones
        self.fc2 = nn.Linear(128, 10)  # 10 classes CIFAR10

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        
        x = F.relu(self.conv3(x))
        x = self.pool(x)
        
        x = x.view(x.size(0), -1)  # aplatit en [batch_size, 64*4*4 = 1024]
        x = F.relu(self.fc1(x)) #ou fonction sigmoïde à la fin?
        x = self.fc2(x)
        return x



à entraîner (boucle d'entraînement) training loop + sauvegarde des poids à chaque epoch ; tester pour chaque epoch et voir où l'accuracy est la mieux

Masque pour le dropout

In [ ]:
class CNN_MCdropout(nn.Module):
    def __init__(self, model, mc_layers=None, p1=0.2, p2=0.2, p3=0.2, p4=0.2):
        super().__init__()
        self.model = model
        self.mc_layers = mc_layers or []

        # Ajout dropout après conv1
        if 'conv1' in self.mc_layers and isinstance(self.model.conv1, nn.Conv2d):
            self.model.conv1 = nn.Sequential(self.model.conv1, nn.ReLU(), nn.Dropout2d(p1))
        else:
            self.model.conv1 = nn.Sequential(self.model.conv1, nn.ReLU())

        # Ajout dropout après conv2
        if 'conv2' in self.mc_layers and isinstance(self.model.conv2, nn.Conv2d):
            self.model.conv2 = nn.Sequential(self.model.conv2, nn.ReLU(), nn.Dropout2d(p2))
        else:
            self.model.conv2 = nn.Sequential(self.model.conv2, nn.ReLU())

        # Ajout dropout après conv3
        if 'conv3' in self.mc_layers and isinstance(self.model.conv3, nn.Conv2d):
            self.model.conv3 = nn.Sequential(self.model.conv3, nn.ReLU(), nn.Dropout2d(p3))
        else:
            self.model.conv3 = nn.Sequential(self.model.conv3, nn.ReLU())

        # Ajout dropout après fc1
        if 'fc1' in self.mc_layers and isinstance(self.model.fc1, nn.Linear):
            self.model.fc1 = nn.Sequential(self.model.fc1, nn.ReLU(), nn.Dropout(p4))
        else:
            self.model.fc1 = nn.Sequential(self.model.fc1, nn.ReLU())

    def forward(self, x):
        x = self.model.conv1(x)
        x = self.model.pool(x)

        x = self.model.conv2(x)
        x = self.model.pool(x)

        x = self.model.conv3(x)
        x = self.model.pool(x)

        x = x.view(x.size(0), -1)
        x = self.model.fc1(x)
        x = self.model.fc2(x)
        return x

In [565]:
# class CNN_MCdropout_generic(nn.Module):
#     def __init__(self, dico_layers):
#         super().__init__()
#         self.mc_layers = dico_layers.get('layers', {})
#         self.conv1 = dico_layers['model'].conv1
#         self.conv2 = dico_layers['model'].conv2
#         self.conv3 = dico_layers['model'].conv3
#         self.pool = dico_layers['model'].pool
#         self.fc1 = dico_layers['model'].fc1
#         self.fc2 = dico_layers['model'].fc2

#     def forward(self, x):
#         # Conv1
#         x = F.relu(self.conv1(x))
#         x = self.pool(x)
#         if 'conv1' in self.mc_layers: 
#             x = F.dropout2d(x, self.mc_layers['conv1'], training=True)

#         # Conv2
#         x = F.relu(self.conv2(x))
#         x = self.pool(x)
#         if 'conv2' in self.mc_layers: 
#             x = F.dropout2d(x, self.mc_layers['conv2'], training=True)

#         # Conv3
#         x = F.relu(self.conv3(x))
#         x = self.pool(x)
#         if 'conv3' in self.mc_layers: 
#             x = F.dropout2d(x, self.mc_layers['conv3'], training=True)

#         # FC1
#         x = torch.flatten(x, 1)
#         x = F.relu(self.fc1(x))
#         if 'fc1' in self.mc_layers: 
#             x = F.dropout(x, self.mc_layers['fc1'], training=True)

#         # FC2 (souvent pas de dropout ici)
#         x = self.fc2(x)

#         return x

In [567]:
# class CNN_MCDropout_Generic(nn.Module):
#     def __init__(self, base_model, mc_layers=None, default_p=0.1):
#         super().__init__()
#         self.base_model = base_model
#         self.mc_layers = mc_layers or {}  # dict {nom_module: p}
#         self.default_p = default_p

#     def apply_mc_dropout(self, module, x, p=None):
#         """Applique MC Dropout sur Conv2d et Linear si spécifié"""
#         if p is None:
#             p = self.default_p
#         if isinstance(module, nn.Conv2d):
#             x = F.dropout2d(x, p=p, training=True)
#         elif isinstance(module, nn.Linear):
#             # Flatten si nécessaire
#             if x.dim() > 2:
#                 x = torch.flatten(x, 1)
#             x = F.dropout(x, p=p, training=True)
#         return x

#     def forward(self, x):
#         return self._forward_recursive(self.base_model, x)

#     def _forward_recursive(self, module, x, prefix=""):
#         # Container de modules (Sequential, ModuleList, ModuleDict)
#         if isinstance(module, (nn.Sequential, nn.ModuleList, nn.ModuleDict)):
#             for name, submodule in module.named_children():
#                 x = self._forward_recursive(submodule, x, prefix + name + ".")
#             return x
#         else:
#             # Couche terminale
#             out = module(x)
#             fullname = prefix.rstrip(".")
#             # Vérifie si on doit appliquer MC Dropout
#             for layer_name, p in self.mc_layers.items():
#                 if fullname.endswith(layer_name):
#                     out = self.apply_mc_dropout(module, out, p)
#             return out

In [568]:
# class CNN_MCdropout_Generic(nn.Module):
#     def __init__(self, model, dico_layers=None):
#         super().__init__()
#         self.model = model
#         self.dico_layers = dico_layers or {}

#         for name, layer in self.model.named_modules():
#             #print(name, flush=True)
#             if isinstance(layer, nn.Conv2d) and name in self.dico_layers:
#                 p = self.dico_layers[name]
#                 self.model.add_module(name, nn.Sequential(layer, nn.ReLU(), nn.Dropout2d(p)))
                
#             elif isinstance(layer, nn.Linear) and name in self.dico_layers:
#                 p = self.dico_layers[name]
#                 self.model.add_module(name, nn.Sequential(layer, nn.ReLU(), nn.Dropout(p)))
                    

#     def forward(self, x):
#         flatten = False
#         for _, layer in self.model.named_modules():
#             if not flatten and isinstance(layer, nn.Linear):
#                 flatten = True
#                 x = x.view(x.size(0), -1)

#             x = layer(x)

#             if isinstance(layer, nn.Conv2d): #verifier que tjrs après conv2d ?!
#                 x = self.model.pool(x)

#         return x

In [569]:
import torch.nn as nn

class CNN_MCdropout_Generic(nn.Module):
    def __init__(self, model, dico_layers=None):
        super().__init__()
        self.model = model
        self.dico_layers = dico_layers or {}

        # Remplacement des couches ciblées
        for name, layer in list(self.model._modules.items()):
            if isinstance(layer, nn.Conv2d) and name in self.dico_layers:
                p = self.dico_layers[name]
                self.model._modules[name] = nn.Sequential(
                    layer, nn.ReLU(), nn.Dropout2d(p)
                )

            elif isinstance(layer, nn.Linear) and name in self.dico_layers:
                p = self.dico_layers[name]
                self.model._modules[name] = nn.Sequential(
                    layer, nn.ReLU(), nn.Dropout(p)
                )

    def forward(self, x):
        return self.model.forward(x)

    

Fonction d'entraînement et test

In [570]:
val_ratio = 0.1  # 10% pour validation
train_size = int((1 - val_ratio) * len(trainset))
val_size   = len(trainset) - train_size

train_subset, val_subset = random_split(trainset, [train_size, val_size]) # divise le dataset de manière aléatoire en 90/10

trainloader = torch.utils.data.DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=2)
valloader   = torch.utils.data.DataLoader(val_subset, batch_size=batch_size, shuffle=False, num_workers=2)

Fonction d'évaluation (avant entraînement)

In [571]:
def evaluate(model, dataloader, device):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for inputs, targets in dataloader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            total_loss += criterion(outputs, targets).item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    return total_loss/len(dataloader), correct/total

Fonction d'entraînement

In [572]:
def train_model(model, trainloader, valloader, device, epochs=20, save_path="best.pt"):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    best_val_acc = 0
    
    for epoch in range(epochs):#petit nombre d'epochs pour tester (environ 1 minutes pas epoch)
        model.train()
        running_loss = 0.0
        for inputs, targets in trainloader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        
        # Évaluer sur validation
        val_loss, val_acc = evaluate(model, valloader, device)
        print(f"Epoch {epoch+1}/{epochs} - Train Loss: {running_loss/len(trainloader):.4f} - Val Loss: {val_loss:.4f} - Val Acc: {val_acc:.4f}")
        
        # Sauvegarde si amélioration
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), save_path)
    
    print("Finished Training")
    return model

In [573]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
save_path = "best.pt"

# Demande à l'utilisateur quelles couches doivent subir le dropout : ou alors il amène sa liste
user_layers = input(
    "Sur quelles couches voulez-vous appliquer le MC Dropout ? "
    "(choisissez parmi conv1, conv2, conv3, fc1, séparées par des virgules) : ")
mc_layers = [layer.strip() for layer in user_layers.split(',') if layer.strip() in ['conv1','conv2','conv3','fc1']]

# Vérifie si les poids existent déjà
base_model = SimpleCNN()
if os.path.exists(save_path):
    print("Chargement du modèle sauvegardé")
    base_model.load_state_dict(torch.load(save_path, map_location=device))  # même architecture que celle qui a sauvegardé
else:
    print("Pas de modèle sauvegardé, on entraîne le modèle")
    base_model = train_model(base_model, trainloader, valloader, device, epochs=20, save_path=save_path)
    base_model.load_state_dict(torch.load(save_path, map_location=device))  # recharge les meilleurs poids

dico_layers = {
    "model": base_model,
    "layers": {
        "conv1": 0.2,
        "conv2": 0.2,
        "conv3": 0.2,
        "fc1": 0.2,
    }
}

# On fait une copie pour MC Dropout
base_model_mc = copy.deepcopy(base_model)

model = CNN_MCdropout(base_model_mc, mc_layers=mc_layers, p1=0.2, p2=0.2, p3=0.2, p4=0.2).to(device)

# Récupérer le dictionnaire des dropouts
mc_layers_dict = dico_layers["layers"]

# Copier le modèle pour le MC Dropout
base_model_dict = copy.deepcopy(dico_layers["model"])

# Créer le modèle MC Dropout générique correctement
model2 = CNN_MCdropout_Generic(base_model_dict, dico_layers=mc_layers_dict).to(device)

# Évaluation finale
test_loss, test_acc = evaluate(model, testloader, device)
print(f"Final Test Loss: {test_loss:.4f} - Test Acc: {test_acc:.4f}")
test_loss2, test_acc2 = evaluate(model2, testloader, device)
print(f"Final Test Loss: {test_loss2:.4f} - Test Acc: {test_acc2:.4f}")


Chargement du modèle sauvegardé
Final Test Loss: 0.8186 - Test Acc: 0.7267
Final Test Loss: 0.8186 - Test Acc: 0.7267


MC Dropout prédiction

In [574]:
def mc_predict_mean_probs(model, X, T=1000, verbose=True):
    model.train()  # activer dropout pendant l'inférence MC
    probs_list = []
    start_time = time.time() 
    with torch.no_grad():
        for _ in tqdm(range(T), disable=not verbose):
            logits_t = model(X)
            probs_t = F.softmax(logits_t, dim=1)
            probs_list.append(probs_t.unsqueeze(0))
    elapsed = time.time() - start_time
    probs_mc = torch.cat(probs_list, dim=0)       
    model.eval()  # remettre le modèle en mode eval à la fin
    if verbose:
        print(f"Temps total: {elapsed:.2f} s  |  Temps moyen par passe: {elapsed/T:.4f} s")
    return probs_mc.mean(0), elapsed                        

je dois garder le même batch ; mettre des seeds pour que ce soit reproductible (fonction de dropout)

In [575]:
# Test MC Dropout sur un batch
X, Y = next(iter(testloader))
X, Y = X.to(device), Y.to(device)
probs, t1 = mc_predict_mean_probs(model, X, T=1000, verbose=True)
probs2, t2 = mc_predict_mean_probs(model2, X, T=1000, verbose=True)
Y_hat = probs.argmax(1)
Y_hat2 = probs2.argmax(1)

print("Classes vraies       :", Y.tolist())
print(f"Classes prédites   (t={t1:.2f}s): {Y_hat.tolist()}")
print(f"Classes prédites 2   (t={t2:.2f}s): {Y_hat2.tolist()}")

acc = (Y_hat == Y).float().mean().item()
print(f"MC Dropout Test Acc: {acc:.4f}")
acc_mc = (Y_hat2 == Y).float().mean().item()
print(f"MC Dropout Générique Test Acc: {acc_mc:.4f}")



100%|██████████| 1000/1000 [00:09<00:00, 100.01it/s]


Temps total: 10.00 s  |  Temps moyen par passe: 0.0100 s


100%|██████████| 1000/1000 [00:13<00:00, 71.64it/s]

Temps total: 13.96 s  |  Temps moyen par passe: 0.0140 s
Classes vraies       : [3, 8, 8, 0, 6, 6, 1, 6, 3, 1, 0, 9, 5, 7, 9, 8, 5, 7, 8, 6, 7, 0, 4, 9, 5, 2, 4, 0, 9, 6, 6, 5, 4, 5, 9, 2, 4, 1, 9, 5, 4, 6, 5, 6, 0, 9, 3, 9, 7, 6, 9, 8, 0, 3, 8, 8, 7, 7, 4, 6, 7, 3, 6, 3, 6, 2, 1, 2, 3, 7, 2, 6, 8, 8, 0, 2, 9, 3, 3, 8, 8, 1, 1, 7, 2, 5, 2, 7, 8, 9, 0, 3, 8, 6, 4, 6, 6, 0, 0, 7, 4, 5, 6, 3, 1, 1, 3, 6, 8, 7, 4, 0, 6, 2, 1, 3, 0, 4, 2, 7, 8, 3, 1, 2, 8, 0, 8, 3]
Classes prédites   (t=10.00s): [3, 8, 8, 0, 6, 6, 1, 6, 3, 1, 0, 9, 5, 7, 9, 8, 5, 7, 8, 6, 7, 2, 4, 9, 4, 2, 4, 0, 9, 6, 3, 5, 4, 3, 9, 3, 4, 1, 9, 5, 4, 6, 3, 6, 0, 9, 3, 3, 7, 6, 9, 8, 6, 3, 8, 8, 5, 5, 5, 3, 7, 5, 6, 3, 6, 6, 1, 0, 3, 7, 0, 6, 8, 8, 0, 2, 2, 3, 3, 8, 8, 1, 1, 7, 2, 9, 2, 8, 8, 9, 0, 3, 8, 6, 4, 6, 6, 0, 0, 3, 4, 5, 6, 3, 1, 1, 2, 6, 8, 5, 5, 0, 2, 2, 9, 3, 0, 4, 3, 5, 8, 3, 1, 2, 8, 0, 8, 3]
Classes prédites 2   (t=13.96s): [3, 8, 8, 0, 6, 6, 1, 6, 3, 1, 0, 9, 5, 7, 9, 8, 5, 7, 8, 6, 7, 2, 4, 1, 4, 2, 4, 0, 9

rajouter en argument de generate_mc_outputs measure = variance (en gros on laisse à l'utilisateur le choix de la mesure d'incertitude)

In [576]:
def generate_mc_outputs(model, X, T=1000, metrics=["mc_estimate"], labels=None, verbose=True):
    model.train()
    outputs = []
    mean_probs = None 

    start_forward = time.time()
    with torch.no_grad():
        for _ in tqdm(range(T), disable=not verbose):
            out = model(X)
            outputs.append(out.unsqueeze(0))
    elapsed_forward = time.time() - start_forward

    outputs = torch.cat(outputs, dim=0)
    results = {}
    elapsed_metrics = {}

    for metric in metrics:
        start_metric = time.time()
        if metric == "mc_estimate":
            all_probs = torch.softmax(outputs, dim=2)
            results[metric] = all_probs.mean(dim=0)
        elif metric == "variance":
            results[metric] = outputs.var(dim=0)
        elif metric == "predictive_entropy":
            all_probs = torch.softmax(outputs, dim=2)
            mean_probs = all_probs.mean(dim=0)
            results[metric] = -(mean_probs * (mean_probs + 1e-12).log()).sum(dim=1)
        elif metric == "relative_norm":
            if labels is None:
                raise ValueError("labels doivent être fournis pour relative_norm")
            all_probs = torch.softmax(outputs, dim=2)
            mean_probs = all_probs.mean(dim=0)
            labels_onehot = F.one_hot(labels, num_classes=mean_probs.size(1)).float()
            diff_norm = torch.norm(mean_probs - labels_onehot, dim=1)
            denom = torch.max(torch.norm(mean_probs, dim=1), torch.norm(labels_onehot, dim=1))
            results[metric] = diff_norm / (denom + 1e-12)
        else:
            raise ValueError(f"Métrique {metric} non reconnue")
        elapsed_metrics[metric] = time.time() - start_metric

    model.eval()

    if verbose:
        print(f"Temps forward pass: {elapsed_forward:.2f}s  |  Temps moyen par passe: {elapsed_forward/T:.4f}s")
        for m, t in elapsed_metrics.items():
            print(f"Temps calcul métrique '{m}': {t:.6f}s")

    return outputs, mean_probs, results, elapsed_forward, elapsed_metrics


le mc estimate n'est pas une mesure d'incertitude, sert pour construire une autre variable

Métriques

In [578]:
user_metrics = input(
    "Quelles métriques voulez-vous calculer ? (mc_estimate, variance, predictive_entropy, relative_norm)\n"
    "Vous pouvez en choisir plusieurs, séparées par des virgules : ")
user_metrics = [m.strip() for m in user_metrics.split(",")]
outputs, mean_probs, metric_values, elapsed_forward, elapsed_metrics = generate_mc_outputs(model, X, T=1000, metrics=user_metrics, labels=Y, verbose=False)

print(f"Liste des métriques choisies par l\'utilisateur : {user_metrics}")
print(f"Temps total des forward passes : {elapsed_forward:.2f} s")
for metric in user_metrics:
    print(f"Métrique choisie : {metric}")
    print(f"Temps de calcul de la métrique : {elapsed_metrics[metric]:.6f} s")
    print(f"Résultat : {metric_values[metric]}\n")

Liste des métriques choisies par l'utilisateur : ['mc_estimate', 'variance', 'predictive_entropy', 'relative_norm']
Temps total des forward passes : 9.38 s
Métrique choisie : mc_estimate
Temps de calcul de la métrique : 0.004488 s
Résultat : tensor([[3.3715e-02, 1.0865e-02, 1.0266e-02,  ..., 1.6014e-03, 1.3793e-01,
         8.6719e-03],
        [1.9850e-02, 2.8709e-01, 7.8428e-04,  ..., 1.0627e-03, 6.6040e-01,
         3.5064e-03],
        [1.5536e-01, 5.1532e-02, 2.5717e-02,  ..., 2.1286e-02, 5.7570e-01,
         3.5096e-02],
        ...,
        [5.9594e-01, 7.3590e-02, 1.9151e-01,  ..., 2.2037e-04, 3.5036e-02,
         3.5282e-02],
        [2.4049e-01, 1.7796e-03, 7.2448e-02,  ..., 2.1173e-03, 4.2845e-01,
         5.6385e-03],
        [1.9984e-02, 3.0764e-03, 3.1760e-02,  ..., 1.1239e-02, 1.0057e-02,
         7.7527e-03]])

Métrique choisie : variance
Temps de calcul de la métrique : 0.003008 s
Résultat : tensor([[ 3.9425,  4.6005,  1.7167,  ...,  4.0787,  9.0217,  5.8706],
        

Métriques générique

In [579]:
user_metrics = input(
    "Quelles métriques voulez-vous calculer ? (mc_estimate, variance, predictive_entropy, relative_norm)\n"
    "Vous pouvez en choisir plusieurs, séparées par des virgules : ")
user_metrics = [m.strip() for m in user_metrics.split(",")]
model2.train()
outputs, mean_probs, metric_values, elapsed_forward, elapsed_metrics = generate_mc_outputs(model2, X, T=1000, metrics=user_metrics, labels=Y, verbose=False)


print(f"Liste des métriques choisies par l\'utilisateur : {user_metrics}")
print(f"Temps total des forward passes : {elapsed_forward:.2f} s")
for metric in user_metrics:
    print(f"Métrique choisie : {metric}")
    print(f"Temps de calcul de la métrique : {elapsed_metrics[metric]:.6f} s")
    print(f"Résultat : {metric_values[metric]}\n")

Liste des métriques choisies par l'utilisateur : ['mc_estimate', 'variance', 'predictive_entropy', 'relative_norm']
Temps total des forward passes : 10.49 s
Métrique choisie : mc_estimate
Temps de calcul de la métrique : 0.004865 s
Résultat : tensor([[0.0698, 0.0292, 0.0555,  ..., 0.0085, 0.1592, 0.0325],
        [0.0636, 0.3471, 0.0151,  ..., 0.0033, 0.4770, 0.0226],
        [0.1627, 0.1012, 0.0578,  ..., 0.0323, 0.3735, 0.0619],
        ...,
        [0.3777, 0.0975, 0.2091,  ..., 0.0041, 0.1027, 0.0701],
        [0.2265, 0.0176, 0.1198,  ..., 0.0120, 0.3128, 0.0204],
        [0.0275, 0.0107, 0.0714,  ..., 0.0455, 0.0221, 0.0341]])

Métrique choisie : variance
Temps de calcul de la métrique : 0.003239 s
Résultat : tensor([[12.2895, 16.8014,  8.7097,  ..., 12.1241, 20.2578, 24.3192],
        [11.7041, 26.1443, 15.1189,  ..., 29.4807, 25.5207, 16.6003],
        [ 5.8933, 14.5679,  7.3901,  ..., 11.7143, 11.7419, 11.6567],
        ...,
        [14.7116, 20.9219, 15.0737,  ..., 22.4345, 1

pour la fonction de confiance, dois-je faire 1/métrique choisie? ou ça marche qu'avec la variance?

oui si c'est dans R+ : 1/ ? exp(-...) si c'est pour avoir entre 0 et 1 ; quitte à normaliser après